In [3]:
from datetime import datetime
import numpy as np

from investment_lab.data.option_db import OptionLoader, extract_spot_from_options
from investment_lab.option_trade import DeltaHedgedOptionTrade
from investment_lab.option_strategies import SHORT_1W_STRADDLE
from investment_lab.backtest import BacktesterBidAskFromData
from investment_lab.stochastic.heston_kalman import HestonUKF
from investment_lab.pricing.implied_volatility import extract_atm_implied_vol
from investment_lab.strategy.dynamic_allocation import compute_linear_weights, rescale_positions_with_signal
from investment_lab.metrics.performance import format_perf_table
#from investment_lab.strategy.plots import plot_results

START  = datetime(2020, 1, 2)
END    = datetime(2022, 12, 30)
TICKER = "SPY"

ModuleNotFoundError: No module named 'investment_lab'

In [ ]:
# ===========================================================================
# [1] Imports & config
# ===========================================================================


# ===========================================================================
# [2] Load options data & log-returns
# ===========================================================================
df_options  = OptionLoader.load_data(start_date=START, end_date=END, process_kwargs={"ticker": TICKER})
df_spot     = extract_spot_from_options(df_options).set_index("date")["spot"]
log_returns = np.log(df_spot / df_spot.shift(1)).dropna()

# ===========================================================================
# [3] Kalman filter — filtered variance v_hat_t
# ===========================================================================
ukf           = HestonUKF(kappa=2.0, theta=0.04, xi=0.30)
kalman_result = ukf.rolling_filter(log_returns, window=63, recalib_every=21)

# ===========================================================================
# [4] ATM implied vol per day
# ===========================================================================
atm_iv = extract_atm_implied_vol(df_options, atm_band=0.02)

# ===========================================================================
# [5] IV-RV spread  s_t = IV_t - sigma_hat_t
# ===========================================================================
spread = HestonUKF.compute_iv_rv_spread(
    implied_vol=atm_iv,
    sigma_filtered=kalman_result["sigma_filtered"],
)

# ===========================================================================
# [6] Dynamic allocation weights  w_t = linear(s_t)  in [0, 1]
# ===========================================================================
dynamic_weights = compute_linear_weights(spread, w_min=0.0, w_max=1.0)

# ===========================================================================
# [7] Delta-hedged positions (static)
# ===========================================================================
df_positions_static  = DeltaHedgedOptionTrade.generate_trades(
    start_date=START, end_date=END, tickers=[TICKER], legs=SHORT_1W_STRADDLE,
)

# ===========================================================================
# [8] Apply dynamic weights
# ===========================================================================
df_positions_dynamic = rescale_positions_with_signal(df_positions_static, dynamic_weights)

# ===========================================================================
# [9] Backtests
# ===========================================================================
backtester_static  = BacktesterBidAskFromData(df_positions_static).compute_backtest()
backtester_dynamic = BacktesterBidAskFromData(df_positions_dynamic).compute_backtest()

# ===========================================================================
# [10] Performance summary
# ===========================================================================
print(format_perf_table({
    "Static (delta-hedged carry)": backtester_static.nav,
    "Dynamic (UKF spread signal)": backtester_dynamic.nav,
}))

# ===========================================================================
# [11] Plots
# ===========================================================================
plot_results(
    backtester_static=backtester_static,
    backtester_dynamic=backtester_dynamic,
    spread=spread,
    dynamic_weights=dynamic_weights,
    atm_iv=atm_iv,
    sigma_filtered=kalman_result["sigma_filtered"],
    log_returns=log_returns,
)

In [ ]:
# ===========================================================================
# [1] Imports & config
# ===========================================================================
from datetime import datetime
import numpy as np

from investment_lab.data.option_db import OptionLoader, extract_spot_from_options
from investment_lab.option_trade import DeltaHedgedOptionTrade
from investment_lab.option_strategies import SHORT_1W_STRADDLE
from investment_lab.backtest import BacktesterBidAskFromData
from investment_lab.stochastic.heston_kalman import HestonUKF
from investment_lab.pricing.implied_volatility import extract_atm_implied_vol
from investment_lab.strategy.dynamic_allocation import compute_linear_weights, rescale_positions_with_signal
from investment_lab.metrics.performance import format_perf_table
from investment_lab.strategy.plots import plot_results

START  = datetime(2020, 1, 2)
END    = datetime(2022, 12, 30)
TICKER = "SPY"

# ===========================================================================
# [2] Load options data & log-returns
# ===========================================================================
df_options  = OptionLoader.load_data(start_date=START, end_date=END, process_kwargs={"ticker": TICKER})
df_spot     = extract_spot_from_options(df_options).set_index("date")["spot"]
log_returns = np.log(df_spot / df_spot.shift(1)).dropna()

# ===========================================================================
# [3] Kalman filter — filtered variance v_hat_t
# ===========================================================================
ukf           = HestonUKF(kappa=2.0, theta=0.04, xi=0.30)
kalman_result = ukf.rolling_filter(log_returns, window=63, recalib_every=21)

# ===========================================================================
# [4] ATM implied vol per day
# ===========================================================================
atm_iv = extract_atm_implied_vol(df_options, atm_band=0.02)

# ===========================================================================
# [5] IV-RV spread  s_t = IV_t - sigma_hat_t
# ===========================================================================
spread = HestonUKF.compute_iv_rv_spread(
    implied_vol=atm_iv,
    sigma_filtered=kalman_result["sigma_filtered"],
)

# ===========================================================================
# [6] Dynamic allocation weights  w_t = linear(s_t)  in [0, 1]
# ===========================================================================
dynamic_weights = compute_linear_weights(spread, w_min=0.0, w_max=1.0)

# ===========================================================================
# [7] Delta-hedged positions (static)
# ===========================================================================
df_positions_static  = DeltaHedgedOptionTrade.generate_trades(
    start_date=START, end_date=END, tickers=[TICKER], legs=SHORT_1W_STRADDLE,
)

# ===========================================================================
# [8] Apply dynamic weights
# ===========================================================================
df_positions_dynamic = rescale_positions_with_signal(df_positions_static, dynamic_weights)

# ===========================================================================
# [9] Backtests
# ===========================================================================
backtester_static  = BacktesterBidAskFromData(df_positions_static).compute_backtest()
backtester_dynamic = BacktesterBidAskFromData(df_positions_dynamic).compute_backtest()

# ===========================================================================
# [10] Performance summary
# ===========================================================================
print(format_perf_table({
    "Static (delta-hedged carry)": backtester_static.nav,
    "Dynamic (UKF spread signal)": backtester_dynamic.nav,
}))

# ===========================================================================
# [11] Plots
# ===========================================================================
plot_results(
    backtester_static=backtester_static,
    backtester_dynamic=backtester_dynamic,
    spread=spread,
    dynamic_weights=dynamic_weights,
    atm_iv=atm_iv,
    sigma_filtered=kalman_result["sigma_filtered"],
    log_returns=log_returns,
)

In [5]:
import sys
print(sys.executable)
print(sys.version)

/Users/marieabbyad/Desktop/M2/env/bin/bin/python
3.13.5 (v3.13.5:6cb20a219a8, Jun 11 2025, 12:23:45) [Clang 16.0.0 (clang-1600.0.26.6)]
